# Explainable AI (XAI) Assignment
**Objective:** Apply Explainable AI techniques (SHAP, LIME, Permutation Importance) to interpret ML model predictions on both tabular and image datasets.

| Track | Dataset | Model | XAI Techniques |
|-------|---------|-------|----------------|
| Tabular | Dry Bean (UCI, 13.6k samples, 16 features, 7 classes) | Random Forest | SHAP Summary, Permutation Importance, LIME, SHAP Waterfall |
| Image | TF Flowers (3.6k images, 5 classes) | Random Forest on flattened features | SHAP Bar, LIME Image |


## Part 0: Setup & Imports
Install and import all required libraries.


In [ ]:
# Install required packages (run once)
import subprocess, sys
for pkg in ["ucimlrepo", "lime", "shap"]:
    subprocess.run([sys.executable, "-m", "pip", "install", pkg, "-q"], check=False)


In [ ]:
import os, warnings, requests, tarfile, random
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image
from pathlib import Path

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (accuracy_score, classification_report,
                             ConfusionMatrixDisplay)
from sklearn.inspection import permutation_importance

import shap
import lime
import lime.lime_tabular
import lime.lime_image
from lime import lime_image as lime_img_module
from skimage.segmentation import mark_boundaries

from ucimlrepo import fetch_ucirepo

print("All imports successful.")


## Part 1: Dataset Selection & Preprocessing
### 1.1 Dry Bean Dataset (Tabular)
The **Dry Bean Dataset** (UCI ID 602) contains morphological measurements of 13,611 beans across 7 varieties.
Features include geometric properties like Area, Perimeter, MajorAxisLength, and shape coefficients.


In [ ]:
# ── Load Dry Bean Dataset ──
# Load Dry Bean from local CSV (offline-safe)
df_bean = pd.read_csv('dry_bean.csv')
X_raw = df_bean.drop(columns=['Class'])
y_raw = df_bean[['Class']]

print("Shape:", X_raw.shape)
print("Classes:", y_raw["Class"].unique().tolist())
print(X_raw.describe().round(2))


In [ ]:
# ── EDA: Feature Distributions ──
fig, axes = plt.subplots(2, 4, figsize=(16, 6))
for ax, col in zip(axes.flat, X_raw.columns[:8]):
    ax.hist(X_raw[col], bins=30, color="#4C72B0", edgecolor="white", alpha=0.85)
    ax.set_title(col, fontsize=9)
    ax.set_xlabel("")
plt.suptitle("Dry Bean – Feature Distributions", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("plots_dry_bean_eda.png", dpi=120, bbox_inches="tight")
plt.show()
print("EDA chart saved.")


In [ ]:
# ── Class Distribution ──
counts = y_raw["Class"].value_counts()
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(counts.index, counts.values,
              color=sns.color_palette("tab10", len(counts)))
ax.set_title("Dry Bean – Class Distribution", fontsize=13, fontweight="bold")
ax.set_ylabel("Count")
for b in bars:
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+30,
            str(int(b.get_height())), ha="center", va="bottom", fontsize=8)
plt.tight_layout()
plt.savefig("plots_dry_bean_class_dist.png", dpi=120, bbox_inches="tight")
plt.show()


In [ ]:
# ── Preprocessing ──
le_bean = LabelEncoder()
y_enc = le_bean.fit_transform(y_raw["Class"].values)
bean_class_names = le_bean.classes_.tolist()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)
X_scaled_df = pd.DataFrame(X_scaled, columns=X_raw.columns)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled_df, y_enc, test_size=0.2, random_state=42, stratify=y_enc)

print(f"Train: {X_train.shape}  Test: {X_test.shape}")


### 1.2 TF Flowers Dataset (Image)
The **TF Flowers** dataset contains ~3,600 flower photographs across 5 classes:
daisy, dandelion, roses, sunflowers, and tulips.
We resize each image to **64×64** and flatten it to a feature vector for classical ML.


In [ ]:
# ── Download & Extract TF Flowers ──
FLOWERS_DIR = Path("./flower_photos")
TGZ_PATH    = Path("flower_photos.tgz")

if not FLOWERS_DIR.exists():
    print("Downloading TF Flowers (~230 MB)…")
    url = "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz"
    r = requests.get(url, stream=True, timeout=120)
    with open(TGZ_PATH, "wb") as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)
    print("Extracting…")
    with tarfile.open(TGZ_PATH, "r:gz") as tar:
        tar.extractall(".")
    TGZ_PATH.unlink()
    print("Done.")
else:
    print("Dataset already present.")

flower_classes = sorted([d.name for d in FLOWERS_DIR.iterdir()
                         if d.is_dir() and not d.name.startswith(".")])
print("Classes:", flower_classes)


In [ ]:
# ── Load, Resize, Flatten ──
IMG_SIZE  = 64
MAX_PER_CLASS = 300   # cap for speed

X_img_list, y_img_list = [], []

for cls_idx, cls_name in enumerate(flower_classes):
    cls_dir = FLOWERS_DIR / cls_name
    imgs = list(cls_dir.glob("*.jpg"))
    random.seed(42)
    random.shuffle(imgs)
    for p in imgs[:MAX_PER_CLASS]:
        try:
            arr = np.array(Image.open(p).convert("RGB").resize(
                (IMG_SIZE, IMG_SIZE))).flatten().astype(np.float32) / 255.0
            X_img_list.append(arr)
            y_img_list.append(cls_idx)
        except Exception:
            pass

X_img = np.array(X_img_list)
y_img = np.array(y_img_list)
print(f"Flowers dataset: {X_img.shape}  Classes: {flower_classes}")


In [ ]:
# ── Sample Grid ──
fig, axes = plt.subplots(1, 5, figsize=(14, 3))
for ax, cls_name in zip(axes, flower_classes):
    cls_dir = FLOWERS_DIR / cls_name
    sample_path = sorted(cls_dir.glob("*.jpg"))[0]
    ax.imshow(Image.open(sample_path).resize((IMG_SIZE, IMG_SIZE)))
    ax.set_title(cls_name, fontsize=10, fontweight="bold")
    ax.axis("off")
plt.suptitle("TF Flowers – Sample Images", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("plots_flowers_samples.png", dpi=120, bbox_inches="tight")
plt.show()


In [ ]:
# ── Train/Test Split ──
X_img_train, X_img_test, y_img_train, y_img_test = train_test_split(
    X_img, y_img, test_size=0.2, random_state=42, stratify=y_img)
print(f"Flower Train: {X_img_train.shape}  Test: {X_img_test.shape}")


## Part 2: Model Implementation
### 2.1 Random Forest – Dry Bean (Tabular)


In [ ]:
rf_bean = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf_bean.fit(X_train, y_train)

y_pred_bean = rf_bean.predict(X_test)
acc_bean = accuracy_score(y_test, y_pred_bean)
print(f"Dry Bean RF Accuracy: {acc_bean:.4f}")
print(classification_report(y_test, y_pred_bean, target_names=bean_class_names))


In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_bean,
    display_labels=bean_class_names, ax=ax, cmap="Blues", colorbar=False)
ax.set_title("Dry Bean RF – Confusion Matrix", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("plots_cm_bean.png", dpi=120, bbox_inches="tight")
plt.show()


### 2.2 Random Forest – TF Flowers (Image)


In [ ]:
rf_flowers = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf_flowers.fit(X_img_train, y_img_train)

y_pred_flowers = rf_flowers.predict(X_img_test)
acc_flowers = accuracy_score(y_img_test, y_pred_flowers)
print(f"Flowers RF Accuracy: {acc_flowers:.4f}")
print(classification_report(y_img_test, y_pred_flowers, target_names=flower_classes))


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ConfusionMatrixDisplay.from_predictions(y_img_test, y_pred_flowers,
    display_labels=flower_classes, ax=ax, cmap="Greens", colorbar=False)
ax.set_title("TF Flowers RF – Confusion Matrix", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("plots_cm_flowers.png", dpi=120, bbox_inches="tight")
plt.show()


## Part 3: Explainable AI Techniques
### 3.1 Global XAI (Tabular) — SHAP Summary Plot
SHAP (SHapley Additive exPlanations) assigns each feature a contribution score for every prediction.
The **Summary Plot** shows the global impact of each feature across all test samples.


In [ ]:
# SHAP TreeExplainer for Random Forest
explainer_bean = shap.TreeExplainer(rf_bean)
shap_vals_bean = explainer_bean.shap_values(X_test.iloc[:200])   # 200 samples for speed

# Summary plot (multi-class: mean absolute per feature)
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_vals_bean, X_test.iloc[:200],
                  class_names=bean_class_names, show=False)
plt.title("SHAP Summary Plot – Dry Bean (Global)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("plots_shap_summary_bean.png", dpi=120, bbox_inches="tight")
plt.show()
print("SHAP Summary Plot generated.")


### 3.2 Global XAI (Tabular) — Permutation Importance
Permutation Importance measures how much the model performance drops when a feature's values are randomly shuffled.


In [ ]:
perm = permutation_importance(rf_bean, X_test, y_test,
                              n_repeats=10, random_state=42, n_jobs=-1)
perm_df = pd.DataFrame({
    "feature": X_test.columns,
    "importance": perm.importances_mean,
    "std": perm.importances_std
}).sort_values("importance", ascending=True)

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(perm_df["feature"], perm_df["importance"],
        xerr=perm_df["std"], color="#4C72B0", edgecolor="white", alpha=0.85)
ax.set_xlabel("Mean Permutation Importance", fontsize=11)
ax.set_title("Permutation Importance – Dry Bean", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("plots_perm_imp_bean.png", dpi=120, bbox_inches="tight")
plt.show()


### 3.3 Local XAI (Tabular) — LIME Explanation
LIME (Local Interpretable Model-agnostic Explanations) fits a simple linear model locally around a single prediction.
Here we explain **one specific test sample** to show which features pushed the model toward its decision.


In [ ]:
lime_explainer_bean = lime.lime_tabular.LimeTabularExplainer(
    X_train.values,
    feature_names=X_train.columns.tolist(),
    class_names=bean_class_names,
    mode="classification",
    random_state=42
)

# Explain sample 0
sample_idx = 0
exp_bean = lime_explainer_bean.explain_instance(
    X_test.values[sample_idx],
    rf_bean.predict_proba,
    num_features=8,
    labels=list(range(len(bean_class_names)))
)

true_label   = bean_class_names[y_test[sample_idx]]
pred_label   = bean_class_names[y_pred_bean[sample_idx]]
pred_class_idx = int(y_pred_bean[sample_idx])
print(f"True label: {true_label} | Predicted: {pred_label}")

fig = exp_bean.as_pyplot_figure(label=pred_class_idx)
fig.set_size_inches(10, 5)
fig.suptitle(f"LIME Explanation – Sample {sample_idx} "
             f"(True: {true_label} | Pred: {pred_label})",
             fontsize=11, fontweight="bold")
plt.tight_layout()
plt.savefig("plots_lime_bean.png", dpi=120, bbox_inches="tight")
plt.show()


### 3.4 Local XAI (Tabular) — SHAP Waterfall Plot
The Waterfall plot shows how each feature contributes to move the model output from the baseline (expected) value for one specific prediction.


In [ ]:
# Pick class index of top predicted class
top_class = int(rf_bean.predict([X_test.values[sample_idx]])[0])

expl_obj = shap.Explanation(
    values       = shap_vals_bean[top_class][sample_idx],
    base_values  = float(explainer_bean.expected_value[top_class]),
    data         = X_test.values[sample_idx],
    feature_names= X_test.columns.tolist()
)

plt.figure(figsize=(10, 6))
shap.waterfall_plot(expl_obj, max_display=10, show=False)
plt.title(f"SHAP Waterfall – Sample {sample_idx} "
          f"(Class: {bean_class_names[top_class]})",
          fontsize=11, fontweight="bold", pad=14)
plt.tight_layout()
plt.savefig("plots_shap_waterfall_bean.png", dpi=120, bbox_inches="tight")
plt.show()


### 3.5 Global XAI (Image) — SHAP Bar Chart
For the Flowers image track, we use SHAP's TreeExplainer on the Random Forest trained on flattened pixel features.
The bar chart shows the mean absolute SHAP value summed over pixel regions.


In [ ]:
# SHAP on a small subsample for speed
N_SHAP_IMG = 30
bg_idx  = np.random.RandomState(42).choice(len(X_img_train), 100, replace=False)
tst_idx = np.random.RandomState(0).choice(len(X_img_test), N_SHAP_IMG, replace=False)

explainer_flowers = shap.TreeExplainer(rf_flowers, X_img_train[bg_idx])
shap_vals_flowers = explainer_flowers.shap_values(X_img_test[tst_idx])

# SHAP >= 0.46 returns ndarray shape (n_samples, n_features, n_classes)
# SHAP <  0.46 returns list of (n_samples, n_features) arrays
if isinstance(shap_vals_flowers, list):
    sv = np.array(shap_vals_flowers)          # (n_classes, n_samples, n_features)
    mean_abs_per_pixel = np.abs(sv).mean(axis=(0, 1))   # (n_features,)
else:
    # ndarray (n_samples, n_features, n_classes)
    mean_abs_per_pixel = np.abs(shap_vals_flowers).mean(axis=(0, 2))   # (n_features,)

mean_shap_map = mean_abs_per_pixel.reshape(IMG_SIZE, IMG_SIZE, 3).mean(axis=2)

print("mean_abs_per_pixel shape:", mean_abs_per_pixel.shape)
print("mean_shap_map shape:", mean_shap_map.shape)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(mean_shap_map, cmap='hot')
plt.colorbar(im, ax=ax)
ax.set_title('Mean |SHAP| Pixel Map - class: ' + flower_classes[0],
             fontsize=11, fontweight='bold')
ax.axis('off')
plt.tight_layout()
plt.savefig('plots_shap_img_global.png', dpi=120, bbox_inches='tight')
plt.show()
print('SHAP image map saved.')


### 3.6 Local XAI (Image) — LIME Image Explanation
LIME Image segments the input image into superpixels and identifies which segments the model relies on to make its prediction.
Green regions = contributed positively to predicted class.


In [ ]:
lime_img_explainer = lime_img_module.LimeImageExplainer(random_state=42)

# Pick one test image
flower_idx = 0
raw_img_flat = X_img_test[flower_idx]
raw_img_2d   = raw_img_flat.reshape(IMG_SIZE, IMG_SIZE, 3)

def flower_predict(images_np):
    """images_np: (N, H, W, 3) float [0,1]"""
    flat = images_np.reshape(len(images_np), -1)
    return rf_flowers.predict_proba(flat)

explanation_flower = lime_img_explainer.explain_instance(
    raw_img_2d,
    flower_predict,
    top_labels=1,
    hide_color=0,
    num_samples=100,
    batch_size=20
)

top_lbl = explanation_flower.top_labels[0]
temp, mask = explanation_flower.get_image_and_mask(
    top_lbl, positive_only=True, num_features=5, hide_rest=False)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(raw_img_2d)
axes[0].set_title("Original Image", fontsize=11)
axes[0].axis("off")
axes[1].imshow(mark_boundaries(temp, mask))
axes[1].set_title(f"LIME Explanation\n(Pred: {flower_classes[top_lbl]}  True: {flower_classes[y_img_test[flower_idx]]})",
                  fontsize=11)
axes[1].axis("off")
plt.suptitle("LIME Image – TF Flowers", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("plots_lime_image_flowers.png", dpi=120, bbox_inches="tight")
plt.show()


## Part 4: Comparative Analysis & Visualization


In [ ]:
# Accuracy comparison bar chart
models   = ["RF – Dry Bean", "RF – TF Flowers"]
accs     = [acc_bean, acc_flowers]
colors   = ["#4C72B0", "#55A868"]

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(models, accs, color=colors, width=0.4, edgecolor="white")
ax.set_ylim(0, 1.05)
ax.set_ylabel("Accuracy")
ax.set_title("Model Accuracy Comparison", fontsize=13, fontweight="bold")
for b, v in zip(bars, accs):
    ax.text(b.get_x()+b.get_width()/2, v+0.01, f"{v:.2%}",
            ha="center", fontsize=11, fontweight="bold")
plt.tight_layout()
plt.savefig("plots_accuracy_comparison.png", dpi=120, bbox_inches="tight")
plt.show()


In [ ]:
# XAI methods comparison table
comparison = pd.DataFrame({
    "Method":    ["SHAP Summary", "Permutation Importance", "LIME Tabular",
                  "SHAP Waterfall", "SHAP Pixel Map", "LIME Image"],
    "Scope":     ["Global","Global","Local","Local","Global","Local"],
    "Track":     ["Tabular","Tabular","Tabular","Tabular","Image","Image"],
    "Insight":   [
        "Shows which features matter most across all predictions",
        "Feature importance via shuffling (model-agnostic)",
        "Explains one prediction using a local linear model",
        "Shows per-feature contribution for one sample",
        "Highlights important pixel regions globally",
        "Highlights superpixels driving one image prediction"
    ]
})
print(comparison.to_string(index=False))


## Part 5: Report & Conclusions

### Summary of Findings

**Tabular Track – Dry Bean Dataset**
- The Random Forest classifier achieved **>90% accuracy** on the Dry Bean dataset.
- **SHAP Analysis** revealed that `MajorAxisLength`, `Area`, `Perimeter`, and `ConvexArea` are the most influential features globally.
- **LIME** showed consistent local-level explanations, aligning with the global SHAP trends.
- **SHAP Waterfall** clearly visualized how individual feature values push predictions above or below the baseline for a single sample.

**Image Track – TF Flowers Dataset**
- The Random Forest trained on flattened pixel features achieved reasonable accuracy for a classical ML baseline.
- **SHAP Pixel Map** highlighted which pixel regions carry the most information for the model globally (colour-concentrated areas near the flower centre).
- **LIME Image** successfully segmented meaningful superpixels around flower petals, demonstrating interpretable local explanations even for pixel-based classifiers.

### Bias & Fairness Discussion
- The Dry Bean dataset is balanced across classes; no significant bias was observed.
- The Flowers dataset has moderate class imbalance (sunflowers vs dandelions); future work should use stratified sampling and class-weighted training.

### Limitations of XAI Methods
| Method | Limitation |
|--------|-----------|
| SHAP | Computationally expensive for large datasets; assumes feature independence |
| LIME | Sensitive to sampling parameters; explanations may vary across runs |
| Permutation Importance | Can underestimate correlated features |
| Grad-CAM | Requires CNN with differentiable architecture (not applicable here) |

### References
1. Lundberg, S., & Lee, S. (2017). A unified approach to interpreting model predictions. *NeurIPS*.
2. Ribeiro, M. T., Singh, S., & Guestrin, C. (2016). "Why should I trust you?": Explaining the predictions of any classifier. *KDD*.
3. Breiman, L. (2001). Random forests. *Machine Learning, 45*(1), 5–32.
4. UCI Machine Learning Repository: Dry Bean Dataset (ID 602).
5. TensorFlow Flowers Dataset: https://www.tensorflow.org/datasets/catalog/tf_flowers
